# 🚊 Прогноз посадок в трамваи — ML-модель в потоковом режиме

Ноутбук строит прогноз `boardings(route, date, hour)` на 01.11–31.12.2025 **только ML-моделями** — без ручных коэффициентов и «математических» профилей. Требования организаторов (см. `CLAUDE.md`):

| Требование | Как выполнено |
|---|---|
| ML-модель, проверяемая в **потоковом режиме** | `StreamingForecaster.ingest(день) → forecast(горизонт)`; прогон «день за днём» с метрикой и latency |
| База — **только данные организаторов** | **Модель A** (LightGBM): признаки только из истории валидаций (лаги/уровни того же дня недели и часа, тренд, горизонт) |
| Внешние данные — **отдельный слой-множитель** | **Модель B** (LightGBM) учит `факт / A` по календарю, погоде, событиям, трафику. Коэффициенты `K_*` **вычисляются моделью** (разложение вклада источников), а не вводятся руками |
| Коэффициенты по умолчанию = 1, если источник недоступен | нейтральные значения признаков → `K = 1` |
| Часы 2–3 — технический шум, окно работы по маршруту | окно работы **оценивается по данным** (доля часа в объёме маршрута) |
| Округление half-up — только при выгрузке | внутри прогноз дробный |
| Воспроизводимость | фиксированный seed, `deterministic=True`, артефакты моделей + контракт признаков, ONNX для JVM |

**Прогноз прямой (direct):** признаки фиксируются на точке прогноза (последний известный день) и применяются к любому целевому дню горизонта 1…61. Рекурсии по собственным прогнозам нет — ошибка не накапливается.

**Данные:** Kaggle-датасет `shotme/moscow-transport` (скачивается `kagglehub`; нужные файлы ищутся автоматически — labels из `dataset.zip` или распакованные, внешние данные по имени). Результаты — в `/content/mostrans_out` (опционально копия на Drive). `Runtime → Run all`. GPU не обязателен: LightGBM на ~0.2–0.5 млн строк работает на CPU за минуты (на A100/T4 — CPU-ядра Colab).

In [ ]:
#@title ⚙️ Конфиг
KAGGLE_DATASET = "shotme/moscow-transport"   #@param {type:"string"}
# Папка с нашими external_data (events/traffic из Telegram-парсеров). Пусто = искать только в датасете.
# Календарь и погода при отсутствии скачиваются из первоисточников автоматически.
EXTERNAL_DIR = ""                             #@param {type:"string"}  # например /content/drive/MyDrive/mostrans/external_data
SAVE_TO_DRIVE = False                         #@param {type:"boolean"}  # True — копировать сабмит и артефакты на Google Drive
DRIVE_OUT_DIR = "/content/drive/MyDrive/mostrans_out"  #@param {type:"string"}
WEATHER_FOR_FORECAST = "fact"  #@param ["fact", "fcst"]  # организаторы разрешили фактическую погоду после 31.10; "fcst" — архив прогнозов (как в проде)

SEED = 42
ROUTES = [1, 5, 7, 11, 12, 17, 25, 26, 28, 50]
HIST_START, HIST_END = "2025-01-01", "2025-10-31"
FORECAST_START, FORECAST_END = "2025-11-01", "2025-12-31"
MIN_HISTORY = 35         # первая точка прогноза — после 5 недель истории

In [ ]:
#@title 📦 Импорты
import os, sys, io, time, json, zipfile, warnings, subprocess
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "lightgbm>=4.3", "kagglehub", "onnxmltools", "onnx"], check=False)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, lightgbm as lgb
from tqdm.auto import tqdm
np.random.seed(SEED)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
print("lightgbm", lgb.__version__, "| pandas", pd.__version__, "| colab:", IN_COLAB, "| cpu:", os.cpu_count())

In [ ]:
#@title 📂 Данные с Kaggle (kagglehub) — структура датасета определяется автоматически
if os.environ.get("LOCAL_DATA_DIR"):                      # локальная проверка ноутбука
    DATA_DIR = Path(os.environ["LOCAL_DATA_DIR"])
else:
    import kagglehub
    try:                                                   # приватный датасет: Colab → 🔑 Secrets: KAGGLE_USERNAME, KAGGLE_KEY
        from google.colab import userdata
        os.environ.setdefault("KAGGLE_USERNAME", userdata.get("KAGGLE_USERNAME"))
        os.environ.setdefault("KAGGLE_KEY", userdata.get("KAGGLE_KEY"))
    except Exception:
        pass                                               # публичный датасет — ключи не нужны
    DATA_DIR = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("DATA_DIR:", DATA_DIR)

def find(name, required=True):
    # файл с точным именем в любой вложенной папке датасета
    hits = sorted(DATA_DIR.rglob(name), key=lambda p: len(p.parts))
    if not hits and required:
        raise FileNotFoundError(f"{name} не найден в {DATA_DIR}")
    return hits[0] if hits else None

class OrgData:
    # данные организаторов: распакованные labels/ или dataset.zip (в т.ч. вложенный в другой zip)
    def __init__(self):
        self.zip = None
        if find("labels_day_train.csv", required=False) is not None:
            print("labels: распакованные файлы"); return
        z = find("dataset.zip", required=False)
        if z is None:                                       # dataset.zip внутри другого архива
            for outer in DATA_DIR.rglob("*.zip"):
                with zipfile.ZipFile(outer) as zz:
                    inner = [n for n in zz.namelist() if n.endswith("dataset.zip")]
                    if inner:
                        z = Path(zz.extract(inner[0], "/content/_org" if IN_COLAB else DATA_DIR / "_org")); break
        if z is None:
            raise FileNotFoundError("не найдены ни labels_day_train.csv, ни dataset.zip")
        self.zip = zipfile.ZipFile(z); print("labels: из", z)

    def open(self, name):                                   # name: 'labels_day_train.csv', 'test_submission.csv'
        if self.zip is None:
            return open(find(name), "rb")
        member = next(n for n in self.zip.namelist() if n.endswith(name))
        return self.zip.open(member)

ORG = OrgData()
lab = pd.concat([pd.read_csv(ORG.open(f"labels_day_{p}.csv"), sep=";") for p in tqdm(["train", "test"], desc="labels")])
lab["date"] = pd.to_datetime(lab["date"])
template = pd.read_csv(ORG.open("test_submission.csv"), sep=";")

OUT = Path("/content/mostrans_out" if IN_COLAB else DATA_DIR / "ml_artifacts")
SUB_DIR = OUT / "submissions"
for d in (OUT, SUB_DIR): d.mkdir(exist_ok=True, parents=True)

# --- тензор истории Y[route, day, hour] (только данные организаторов)
HIST_DATES = pd.date_range(HIST_START, HIST_END)
ALL_DATES = pd.date_range(HIST_START, FORECAST_END)
R_IDX = {r: i for i, r in enumerate(ROUTES)}
D_IDX = {d: i for i, d in enumerate(ALL_DATES)}
Y = np.zeros((len(ROUTES), len(HIST_DATES), 24), dtype=np.float32)
Y[lab.route.map(R_IDX).values, lab.date.map(D_IDX).values, lab.hour.values] = lab.boardings.values
DOW = ALL_DATES.dayofweek.values
print(f"Y: {Y.shape} (маршруты × дни × часы), посадок всего {Y.sum():,.0f}")

# --- внешний слой (отдельно, со ссылками — см. EXTERNAL_DATA.md)
def content_find(name):
    # файлы, загруженные прямо в Colab (/content), без обхода Drive и кэша
    root = Path("/content")
    if not root.exists(): return None
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in ("drive", "sample_data", ".config") and not d.startswith(".")]
        if name in filenames: return Path(dirpath) / name
    return None

def ext_find(name):
    # внешний файл: Kaggle-датасет → /content (загружено в Colab) → EXTERNAL_DIR (Drive монтируется при необходимости)
    hit = find(name, required=False) or content_find(name)
    if hit is None and EXTERNAL_DIR:
        if EXTERNAL_DIR.startswith("/content/drive") and IN_COLAB and not Path("/content/drive/MyDrive").exists():
            from google.colab import drive; drive.mount("/content/drive")
        hits = sorted(Path(EXTERNAL_DIR).rglob(name)) if Path(EXTERNAL_DIR).exists() else []
        hit = hits[0] if hits else None
    return hit

import urllib.request, xml.etree.ElementTree as ET
def fetch_calendar():
    # производственный календарь РФ: https://github.com/xmlcalendar/data
    url = "https://raw.githubusercontent.com/xmlcalendar/data/master/ru/2025/calendar.xml"
    root = ET.fromstring(urllib.request.urlopen(url, timeout=60).read())
    c = pd.DataFrame(index=pd.date_range("2025-01-01", "2025-12-31")); c.index.name = "date"
    c["dow"] = c.index.dayofweek
    for col in ("is_holiday", "is_short_workday", "is_working_weekend"): c[col] = 0
    for d in root.iter("day"):
        dt = pd.Timestamp(f"2025-{d.get('d').replace('.', '-')}")
        if d.get("t") == "1": c.loc[dt, "is_holiday"] = 1
        elif d.get("t") == "2":
            c.loc[dt, "is_short_workday"] = 1
            if dt.dayofweek >= 5: c.loc[dt, "is_working_weekend"] = 1
        elif d.get("t") == "3": c.loc[dt, "is_working_weekend"] = 1
    c["is_day_off"] = (((c.dow >= 5) & (c.is_working_weekend == 0)) | (c.is_holiday == 1)).astype(int)
    c["days_to_new_year"] = (pd.Timestamp("2026-01-01") - c.index).days
    print("календарь: скачан из", url)
    return c

def fetch_weather(kind):
    # Open-Meteo: archive (факт, ERA5/IFS) или historical-forecast (архив прогнозов); бесплатно, без ключа
    host = "archive-api.open-meteo.com/v1/archive" if kind == "fact" else "historical-forecast-api.open-meteo.com/v1/forecast"
    url = (f"https://{host}?latitude=55.7558&longitude=37.6173&start_date=2025-01-01&end_date=2025-12-31"
           "&hourly=temperature_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,cloud_cover,wind_speed_10m,weather_code"
           "&timezone=Europe%2FMoscow")
    h = json.loads(urllib.request.urlopen(url, timeout=120).read())["hourly"]
    w = pd.DataFrame(h); w["time"] = pd.to_datetime(w.time)
    w.insert(0, "date", w.time.dt.normalize()); w.insert(1, "hour", w.time.dt.hour)
    print(f"погода ({kind}): скачана из https://{host}")
    return w.drop(columns="time")

f = ext_find("calendar_2025.csv")
cal = pd.read_csv(f, sep=";", parse_dates=["date"]).set_index("date") if f else fetch_calendar()
f = ext_find("weather_fact_hourly_2025.csv")
w_fact = pd.read_csv(f, sep=";", parse_dates=["date"]) if f else fetch_weather("fact")
f = ext_find("weather_fcst_hourly_2025.csv")
w_fcst = pd.read_csv(f, sep=";", parse_dates=["date"]) if f else fetch_weather("fcst")

EXT_AVAILABLE = {"calendar": True, "weather": True}
print("календарь и погода:", cal.shape, w_fact.shape, w_fcst.shape)

In [ ]:
#@title 📰 События и трафик: файл (датасет / Colab / EXTERNAL_DIR) или парсинг Telegram Дептранса
# Источники: https://t.me/s/DtRoad (события, перекрытия, работы на путях), https://t.me/s/DtOperativno (баллы пробок ЦОДД)
import re, html
PARSE_IF_MISSING = True        # нет файлов → парсим публичную веб-версию каналов (~8–12 мин на канал, кэш в OUT/external_data)
EXT_CACHE = OUT / "external_data"; EXT_CACHE.mkdir(exist_ok=True, parents=True)

TARGET_ROUTES = {1, 5, 7, 11, 12, 17, 25, 26, 28, 50}
MONTHS = {"январ": 1, "феврал": 2, "март": 3, "апрел": 4, "ма": 5, "июн": 6, "июл": 7, "август": 8,
          "сентябр": 9, "октябр": 10, "ноябр": 11, "декабр": 12}
MONTH_RE = r"(январ[ья]|феврал[ья]|марта?|апрел[ья]|мая|июн[ья]|июл[ья]|августа?|сентябр[ья]|октябр[ья]|ноябр[ья]|декабр[ья])"

CATEGORIES = {  # категория -> регулярка (по lower-тексту)
    "closure": r"(перекро|перекрыт|закрыт\w* движени|закроют|ограничен\w* движени|временно закрыт)",
    "tram_works": r"(трамва\w*.{0,80}(ремонт|путев|путей|реконструкц|работ|изменени|не будут ходить|приостанов|отмен|компенсационн)|(ремонт|путев|путей|реконструкц).{0,80}трамва)",
    "tram_route_change": r"(трамва\w*.{0,60}(маршрут|№)|маршрут\w* трамва)",
    "mass_event": r"(забег|марафон|парад|фестивал|концерт|матч|выпускн|праздничн\w* мероприяти|салют|велопарад)",
    "weather_alert": r"(снегопад|ливень|сильный дождь|гололед|метел|аномальн\w* (холод|жар)|оранжев\w* уровень|желт\w* уровень)",
    "new_line": r"(нов\w* трамвайн\w* лини|открыл\w*.{0,40}трамва|запуст\w*.{0,40}трамва)",
}


# ---------------------------------------------------------------- crawl
def fetch_page(channel, before=None):
    url = f"https://t.me/s/{channel}" + (f"?before={before}" if before else "")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    t = urllib.request.urlopen(req, timeout=30).read().decode("utf-8", "replace")
    posts = []
    for blk in t.split('class="tgme_widget_message_wrap')[1:]:
        i = re.search(r'data-post="[^/]+/(\d+)"', blk)
        d = re.search(r'datetime="([^"]+)"', blk)
        m = re.search(r'(?s)tgme_widget_message_text[^>]*>(.*?)</div>', blk)
        if not (i and d):
            continue
        txt = ""
        if m:
            txt = re.sub(r"<br\s*/?>", "\n", m.group(1))
            txt = html.unescape(re.sub(r"<[^>]+>", "", txt)).strip()
        posts.append({"id": int(i.group(1)), "datetime_utc": d.group(1), "text": txt,
                      "url": f"https://t.me/{channel}/{i.group(1)}"})
    return posts


def mentioned_dates(text, year):
    ''''с 12 по 14 июля', '12 и 13 июля', '12 июля' -> список дат.'''
    out = set()
    low = text.lower()
    for a, b, mon in re.findall(r"с\s+(\d{1,2})\s+(?:по|до)\s+(\d{1,2})\s+" + MONTH_RE, low):
        m = next(v for k, v in MONTHS.items() if mon.startswith(k))
        for d in range(int(a), int(b) + 1):
            try:
                out.add(pd.Timestamp(year, m, d))
            except ValueError:
                pass
    for d, mon in re.findall(r"(\d{1,2})\s+" + MONTH_RE, low):
        m = next(v for k, v in MONTHS.items() if mon.startswith(k))
        try:
            out.add(pd.Timestamp(year, m, int(d)))
        except ValueError:
            pass
    return sorted(x.strftime("%Y-%m-%d") for x in out)


def tram_routes(text):
    low = text.lower()
    if "трамва" not in low:
        return []
    nums = set()
    # «№ 7», «№7 и 50», «маршрутах 7, 50 и 12», «17-й трамвай», «трамваи 7 и 50»
    for lst in re.findall(r"(?:№\s*|маршрут\w*\s+(?:№\s*)?|трамва\w*\s+(?:№\s*)?)(\d{1,2}(?:\s*(?:,|и)\s*(?:№\s*)?\d{1,2})*)(?!\s*(?:\d|тыс|млн|км|мин|%|ваг|ретро|плат|дн|год|лет|раз|час))", low):
        nums |= {int(n) for n in re.findall(r"\d{1,2}", lst)}
    nums |= {int(n) for n in re.findall(r"\b(\d{1,2})-?(?:й|го|му|м)\s+(?:маршрут|трамва)", low)}
    return sorted(nums & TARGET_ROUTES)



def crawl_channel(channel, start="2025-01-01", end="2025-12-31", sleep=0.7):
    # листаем t.me/s/<channel>?before=<id> назад от конца периода до начала; повтор при пустой странице/ошибке
    start_ts, end_ts = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1)
    lo, hi = 1, fetch_page(channel)[-1]["id"] + 1
    while hi - lo > 20:
        mid = (lo + hi) // 2; pg = fetch_page(channel, mid)
        lo, hi = (mid, hi) if (not pg or pd.Timestamp(pg[-1]["datetime_utc"]) < end_ts) else (lo, mid)
        time.sleep(sleep)
    before, seen, fails = hi + 20, {}, 0
    bar = tqdm(desc=f"@{channel}", unit="стр")
    while True:
        try:
            pg = fetch_page(channel, before)
        except Exception:
            pg = []
        if not pg:
            fails += 1
            if fails > 5: break
            time.sleep(5 * fails); continue
        fails = 0
        for x in pg: seen[x["id"]] = x
        oldest = min(pg, key=lambda x: x["id"]); bar.update(1)
        bar.set_postfix(date=oldest["datetime_utc"][:10], posts=len(seen))
        if pd.Timestamp(oldest["datetime_utc"]) < start_ts: break
        before = oldest["id"]; time.sleep(sleep)
    bar.close()
    df_ = pd.DataFrame(seen.values())
    df_["dt"] = pd.to_datetime(df_.datetime_utc, utc=True).dt.tz_convert("Europe/Moscow")
    return df_[(df_.dt >= pd.Timestamp(start, tz="Europe/Moscow")) & (df_.dt < pd.Timestamp(end, tz="Europe/Moscow") + pd.Timedelta(days=1))]

def extract_events(posts):
    ev = []
    for _, r in posts.iterrows():
        l = r.text.lower()
        cats = [c for c, rx in CATEGORIES.items() if re.search(rx, l)]
        if cats:
            ev.append({"post_date": r["dt"].strftime("%Y-%m-%d"), "categories": "|".join(cats),
                       "target_routes": "|".join(map(str, tram_routes(r.text))),
                       "event_dates": "|".join(mentioned_dates(r.text, r["dt"].year)), "url": r.url})
    return pd.DataFrame(ev, columns=["post_date", "categories", "target_routes", "event_dates", "url"])

def extract_traffic(posts):
    tr = []
    for _, r in posts.iterrows():
        l = r.text.lower()
        if not re.search(r"(цодд|дорог|пробк|движени|загруженност)", l): continue
        for pre, sc in re.findall(r"([^.\n]{0,60}?)(\d{1,2})\s*балл", l):
            if 0 <= int(sc) <= 10 and "парков" not in pre:
                tr.append({"date": r["dt"].strftime("%Y-%m-%d"), "hour": r["dt"].hour, "score": int(sc), "url": r.url})
    return pd.DataFrame(tr, columns=["date", "hour", "score", "url"]).drop_duplicates(["url", "score"])

def load_or_parse(fname, channel, extractor):
    f = ext_find(fname) or (EXT_CACHE / fname if (EXT_CACHE / fname).exists() else None)
    if f is not None:
        print(f"{fname}: файл {f}"); return pd.read_csv(f, sep=";", dtype=str).fillna("")
    if not PARSE_IF_MISSING:
        return None
    print(f"{fname}: файла нет → парсим https://t.me/s/{channel}")
    try:
        posts = crawl_channel(channel)
        out = extractor(posts); out.to_csv(EXT_CACHE / fname, sep=";", index=False)
        print(f"   {len(posts)} постов → {len(out)} записей, сохранено в {EXT_CACHE / fname}")
        return out.astype(str)
    except Exception as e:
        print(f"   парсинг не удался ({type(e).__name__}: {e}) → источник отключён"); return None

events = load_or_parse("events_2025.csv", "DtRoad", extract_events)
EXT_AVAILABLE["events"] = events is not None and len(events) > 0
if not EXT_AVAILABLE["events"]:
    events = pd.DataFrame(columns=["categories", "target_routes", "event_dates", "post_date"])

tr_files = [x for x in [ext_find("DtOperativno_traffic_scores_2025.csv"), ext_find("traffic_scores_2025.csv")] if x is not None]
if tr_files:
    traffic = pd.concat([pd.read_csv(x, sep=";") for x in tr_files], ignore_index=True); print("трафик: файлы", tr_files)
else:
    t_ = load_or_parse("DtOperativno_traffic_scores_2025.csv", "DtOperativno", extract_traffic)
    traffic = t_.assign(score=pd.to_numeric(t_.score)) if t_ is not None else pd.DataFrame(columns=["date", "score"])
EXT_AVAILABLE["traffic"] = len(traffic) > 0
for g, ok in EXT_AVAILABLE.items():
    print(f"{'✅' if ok else '⚠️'} {g}: {'используется' if ok else 'недоступен → K = 1'}")

## 🔎 EDA (кратко)

In [ ]:
#@title Объём маршрутов, суточный профиль, режимы
tot = pd.Series(Y.sum((1, 2)), index=ROUTES, name="посадок")
print((tot / 1e6).round(2).to_frame("млн").T)
fig, ax = plt.subplots(1, 2, figsize=(16, 3.5))
ax[0].bar(range(24), Y.sum((0, 1))); ax[0].set_title("Посадки по часам")
wk = pd.DataFrame(Y.sum(2).T, index=HIST_DATES, columns=ROUTES).resample("W").sum().iloc[1:-1]
wk.plot(ax=ax[1], title="По неделям", legend=False); plt.show()
dd = pd.DataFrame(Y.sum(2).T, index=HIST_DATES, columns=ROUTES)
we = dd[dd.index.dayofweek >= 5].groupby(dd[dd.index.dayofweek >= 5].index.month).sum() / dd.groupby(dd.index.month).sum()
print("Доля выходных в объёме, % (смена режимов 7 и 50 с сентября):"); print((100 * we).round(1))

In [ ]:
#@title Окно работы маршрутов — по данным (технические часы)
share = Y[:, -56:, :].sum(1) / np.maximum(Y[:, -56:, :].sum((1, 2))[:, None], 1)
ACTIVE = share >= 5e-4          # час активен, если ≥0.05% дневного объёма маршрута за последние 8 недель
print(pd.DataFrame(np.where(ACTIVE, "■", "·"), index=ROUTES, columns=range(24)).to_string())
print("Неактивные часы обнуляются в прогнозе; вычислено по данным, а не задано руками.")

## 🧱 Модель A — база на данных организаторов (LightGBM на лаговых признаках)

Для точки прогноза `o` (последний известный день) и целевого дня `t`:

| признак | смысл |
|---|---|
| `pool_med28`, `pool_mean28`, `pool_med14`, `pool_mean14` | посадки в этом **часе** в днях той же **группы** (пн–чт / пт / сб / вс по дате) за последние 28/14 дней — медиана/среднее |
| `lvl_mean4`, `lvl_med2` | тот же час в последние 4/2 дня **с тем же днём недели** |
| `route`, `hour`, `dow` | контекст |

Целевая переменная — отношение `y / (pool_med28 + 1)`, вес — `pool_med28 + 1` (⇒ оптимизация L1 по посадкам = WAPE).

**Почему A учится только на горизонтах h ≤ 3** — это вывод из экспериментов, а не ручная настройка: при обучении на всех горизонтах модель выучивает средний сезонный дрейф января–августа (весна → летний спад) и переносит его на осень (F2 −1.4 п.п.). Модель коротких горизонтов учит только «как день выглядит относительно недавнего уровня», а сезонные и календарные эффекты — задача слоя B. Применяется к любому горизонту: признаки фиксируются на точке прогноза (прямой прогноз, без рекурсии).

In [ ]:
#@title Признаки модели A
GRP = np.array([0, 0, 0, 0, 1, 2, 3])      # группы дней по дате: пн–чт, пт, сб, вс
A_FEATS = ["route", "hour", "dow", "pool_med28", "pool_mean28", "pool_med14", "pool_mean14", "lvl_mean4", "lvl_med2"]
A_REF = "pool_med28"
A_TRAIN_MAX_H = 3

def a_rows(Yh, o, target_idx):
    # признаки для целевых дней target_idx от точки прогноза o (только история до o включительно)
    R, H = len(ROUTES), 24
    tg_all = np.asarray(list(target_idx))
    pools = {}
    for win in (28, 14):
        a = max(0, o - win + 1); blk = Yh[:, a:o + 1, :]; g = GRP[DOW[a:o + 1]]
        for k in range(4):
            sel = blk[:, g == k, :]
            pools[(win, k)] = (np.median(sel, 1), sel.mean(1)) if sel.shape[1] else (np.zeros((R, H)), np.zeros((R, H)))
    dw_state = {}
    for dw in range(7):
        last = o - ((DOW[o] - dw) % 7)
        idx = [last - 7 * k for k in range(4) if last - 7 * k >= 0]
        dw_state[dw] = (Yh[:, idx, :].mean(1), np.median(Yh[:, idx[:2], :], 1))
    out = []
    for t in tg_all:
        dw = DOW[t]; k = GRP[dw]
        out.append(pd.DataFrame({
            "route": np.repeat(ROUTES, H), "hour": np.tile(np.arange(H), R), "dow": dw,
            "pool_med28": pools[(28, k)][0].ravel(), "pool_mean28": pools[(28, k)][1].ravel(),
            "pool_med14": pools[(14, k)][0].ravel(), "pool_mean14": pools[(14, k)][1].ravel(),
            "lvl_mean4": dw_state[dw][0].ravel(), "lvl_med2": dw_state[dw][1].ravel(),
            "h": t - o, "t": t, "o": o}))
    return pd.concat(out, ignore_index=True)

def a_dataset(Yh, cutoff_idx, origins, max_h):
    parts = []
    for o in origins:
        d = a_rows(Yh, o, range(o + 1, min(o + max_h, cutoff_idx) + 1))
        d["y"] = Yh[d.route.map(R_IDX).values, d.t.values, d.hour.values]
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

A_PARAMS = dict(objective="l1", learning_rate=0.05, num_leaves=63, min_data_in_leaf=200, feature_fraction=0.9,
                bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, seed=SEED, deterministic=True,
                force_row_wise=True, verbose=-1, num_threads=os.cpu_count())
A_ROUNDS = 400

def train_lgb(X, y, params, rounds, weight=None, desc="lgb"):
    X = X.copy(); X["route"] = pd.Categorical(X["route"], categories=ROUTES)
    ds = lgb.Dataset(X, y, weight=weight, categorical_feature=["route"], free_raw_data=False)
    bar = tqdm(total=rounds, desc=desc, leave=False)
    m = lgb.train(params, ds, rounds, callbacks=[lambda env: bar.update(1)]); bar.close()
    return m

def lgb_predict(m, X):
    X = X.copy(); X["route"] = pd.Categorical(X["route"], categories=ROUTES)
    return m.predict(X)

def fit_A(d):
    ref = d[A_REF] + 1
    return train_lgb(d[A_FEATS], (d.y / ref).clip(0, 5).values, A_PARAMS, A_ROUNDS, weight=ref.values, desc="A")

def predict_A(m, rows):
    return np.maximum(lgb_predict(m, rows[A_FEATS]) * (rows[A_REF] + 1), 0)

def wape_score(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return max(0.0, 1 - np.abs(y - p).sum() / y.sum())

_demo = a_dataset(Y, D_IDX[pd.Timestamp("2025-08-31")], range(MIN_HISTORY, D_IDX[pd.Timestamp("2025-08-31")]), A_TRAIN_MAX_H)
print(f"выборка A (cutoff 31.08, h ≤ {A_TRAIN_MAX_H}, точка прогноза — каждый день): {len(_demo):,} строк"); print(_demo.head(3))

## 🧩 Модель B — внешний слой

Цель `y / ŷ_A` (вес `ŷ_A`), где `ŷ_A` — **кросс-фит** прогноз базы на горизонтах до 28 дней (A, обученная на другой половине точек прогноза). Признаки — источники критерия 2а, в виде, не протекающем сезонностью:

| группа | признаки | зачем так |
|---|---|---|
| `calendar` | праздник, сокращённый день, рабочий выходной | школьные каникулы исключены: в обучении они почти всегда = лето и путаются с сезоном (проверено экспериментом) |
| `weather` | **аномалия** температуры и снежного покрова к среднему за 28 предыдущих дней, осадки, снегопад | абсолютная температура кодирует сезон; монотонные ограничения: теплее → не меньше, осадки → не больше |
| `events` | работы/перекрытия с упоминанием маршрута, общегородские перекрытия, массовые события | t.me/DtRoad |
| `traffic` | максимальный балл ЦОДД за день | t.me/DtOperativno |

`K_g = B(группы ≤ g реальные) / B(группы < g реальные)` при остальных нейтральных; `Π K_g = B(всё)/B(нейтрально)` — чистая поправка внешних данных, без калибровки базы. Нейтральные значения = «источник недоступен» ⇒ `K = 1`.

In [ ]:
#@title Внешние признаки
CAL_F = ["is_holiday", "is_short_workday", "is_working_weekend"]
W_F = ["temp_anom", "precipitation", "snowfall", "snow_depth_anom"]
EV_F = ["ev_route_works", "ev_city_closure", "ev_mass_event"]
TR_F = ["traffic_max"]
GROUPS = {g: cols for g, cols in {"calendar": CAL_F, "weather": W_F, "events": EV_F, "traffic": TR_F}.items() if EXT_AVAILABLE[g]}
print("внешние источники в модели B:", list(GROUPS))
B_FEATS = ["route", "hour", "dow"] + sum(GROUPS.values(), [])
MONO = {"temp_anom": 1, "precipitation": -1}

ev_route, ev_city, ev_mass = set(), set(), set()
for _, r in events.iterrows():
    for d in (r.event_dates.split("|") if r.event_dates else [r.post_date]):
        d = pd.Timestamp(d)
        if "closure" in r.categories: ev_city.add(d)
        if "mass_event" in r.categories: ev_mass.add(d)
        if r.target_routes and ("tram_works" in r.categories or "closure" in r.categories):
            for rt in r.target_routes.split("|"): ev_route.add((int(rt), d))
TRAFFIC_MAX = traffic.assign(date=pd.to_datetime(traffic.date)).groupby("date").score.max() if len(traffic) else pd.Series(dtype=float)

def weather_anom(kind):
    w = (w_fact if kind == "fact" else w_fcst).copy()
    dly = w.groupby("date").agg(t=("temperature_2m", "mean"), sd=("snow_depth", "mean"))
    dly["t_norm"] = dly.t.shift(1).rolling(28, min_periods=7).mean()
    dly["sd_norm"] = dly.sd.shift(1).rolling(28, min_periods=7).mean()
    w = w.merge(dly[["t_norm", "sd_norm"]], left_on="date", right_index=True)
    w["temp_anom"] = w.temperature_2m - w.t_norm
    w["snow_depth_anom"] = w.snow_depth - w.sd_norm
    return w.set_index(["date", "hour"])[W_F]
W_TRAIN = weather_anom("fact")
W_FC = weather_anom(WEATHER_FOR_FORECAST)

def b_frame(base, weather):
    X = base[["route", "hour"]].copy(); X["dow"] = base.date.dt.dayofweek.values
    c = cal.reindex(base.date)
    for col in CAL_F: X[col] = c[col].values
    w = weather.reindex(pd.MultiIndex.from_arrays([base.date, base.hour]))
    for col in W_F: X[col] = w[col].values
    X["ev_route_works"] = [int((r, d) in ev_route) for r, d in zip(base.route, base.date)]
    X["ev_city_closure"] = base.date.isin(ev_city).astype(int).values
    X["ev_mass_event"] = base.date.isin(ev_mass).astype(int).values
    X["traffic_max"] = TRAFFIC_MAX.reindex(base.date).values.astype(float) if len(TRAFFIC_MAX) else np.nan
    return X[B_FEATS]

NEUTRAL = {**{c: 0 for c in CAL_F}, **{c: 0.0 for c in W_F}, **{c: 0 for c in EV_F}, "traffic_max": np.nan}

def b_params(cols):
    p = dict(objective="regression", learning_rate=0.03, num_leaves=15, min_data_in_leaf=1000, feature_fraction=0.9,
             bagging_fraction=0.8, bagging_freq=1, lambda_l2=10.0, seed=SEED, deterministic=True,
             force_row_wise=True, verbose=-1, num_threads=os.cpu_count())
    p["monotone_constraints"] = [MONO.get(c, 0) for c in cols]
    return p
B_ROUNDS = 300

In [ ]:
#@title Модель: A + кросс-фит + B, прогноз с разложением на K
CACHE = {}

class TramForecaster:
    def __init__(self, groups=tuple(GROUPS)):
        self.groups = list(groups)

    def fit(self, Yh, cutoff_idx):
        key = (cutoff_idx, Yh.shape[1], float(Yh.sum()))
        if key not in CACHE:                  # A и кросс-фит не зависят от набора внешних групп
            dA = a_dataset(Yh, cutoff_idx, range(MIN_HISTORY, cutoff_idx), A_TRAIN_MAX_H)
            A = fit_A(dA)
            origins = list(range(MIN_HISTORY, cutoff_idx, 3))
            dB = a_dataset(Yh, cutoff_idx, origins, 28)
            half = origins[len(origins) // 2]
            oof = np.zeros(len(dB))
            for tr_mask, te_mask in ((dA.o >= half, dB.o < half), (dA.o < half, dB.o >= half)):
                if tr_mask.sum() > 1000 and te_mask.any():
                    oof[te_mask.values] = predict_A(fit_A(dA[tr_mask]), dB[te_mask])
            dB["a"] = oof; dB["date"] = ALL_DATES[dB.t.values]
            dB = dB[dB.a > 1]
            CACHE[key] = (A, dB, b_frame(dB, W_TRAIN))
        self.A, dB, XB = CACHE[key]
        self.B = None
        if self.groups:
            self.cols = ["route", "hour", "dow"] + sum([GROUPS[g] for g in self.groups], [])
            self.B = train_lgb(XB[self.cols], (dB.y / dB.a).clip(0, 3).values, b_params(self.cols), B_ROUNDS,
                               weight=dB.a.values, desc="B")
        return self

    def _neutral(self, X, keep):
        X = X.copy()
        for g in self.groups:
            if g not in keep:
                for c in GROUPS[g]: X[c] = NEUTRAL[c]
        return X

    def predict(self, Yh, origin_idx, target_idx, weather=None):
        rows = a_rows(Yh, origin_idx, target_idx)
        out = pd.DataFrame({"route": rows.route, "date": ALL_DATES[rows.t.values], "hour": rows.hour})
        active = ACTIVE[out.route.map(R_IDX).values, out.hour.values]
        out["base"] = np.where(active, predict_A(self.A, rows), 0.0)
        for g in GROUPS: out[f"K_{g}"] = 1.0
        if self.B is not None:
            X = b_frame(out, W_FC if weather is None else weather)[self.cols]
            prev = np.clip(lgb_predict(self.B, self._neutral(X, [])), 0.2, 3)
            keep = []
            for g in self.groups:
                keep.append(g)
                cur = np.clip(lgb_predict(self.B, self._neutral(X, keep)), 0.2, 3)
                out[f"K_{g}"] = cur / prev; prev = cur
        out["pred"] = out.base * np.prod([out[f"K_{g}"] for g in GROUPS], axis=0)
        return out

def fold_eval(groups, cut, a, b):
    ci, ai, bi = D_IDX[pd.Timestamp(cut)], D_IDX[pd.Timestamp(a)], D_IDX[pd.Timestamp(b)]
    m = TramForecaster(groups).fit(Y[:, :ci + 1], ci)
    p = m.predict(Y[:, :ci + 1], ci, range(ai, bi + 1), weather=W_TRAIN if WEATHER_FOR_FORECAST == "fact" else weather_anom("fcst"))
    p["y"] = Y[p.route.map(R_IDX).values, p.date.map(D_IDX).values, p.hour.values]
    return m, p

In [ ]:
#@title Валидация: 3 хронологических фолда (прогресс и метрика по ходу)
# F1: переход лето→осень; F2: самый близкий аналог ноя–дек (стабильный сезон); F4: праздники 12–13.06 при майских в обучении
FOLDS = [("F1 авг→сен–окт", "2025-08-31", "2025-09-01", "2025-10-31"),
         ("F2 сен→окт",     "2025-09-30", "2025-10-01", "2025-10-31"),
         ("F4 июн→июн–авг", "2025-06-08", "2025-06-09", "2025-08-08")]
rows, FOLD_PRED = [], {}
for name, cut, a, b in tqdm(FOLDS, desc="folds"):
    t0 = time.time()
    m, p = fold_eval(list(GROUPS), cut, a, b); FOLD_PRED[name] = p
    ci = D_IDX[pd.Timestamp(cut)]
    rr = a_rows(Y[:, :ci + 1], ci, range(D_IDX[pd.Timestamp(a)], D_IDX[pd.Timestamp(b)] + 1))
    naive = np.where(ACTIVE[rr.route.map(R_IDX).values, rr.hour.values], rr.pool_med28, 0)
    r = {"fold": name, "naive (медиана 28д, для сравнения)": wape_score(p.y, naive),
         "A (база)": wape_score(p.y, p.base), "A×B (все источники)": wape_score(p.y, p.pred), "сек": round(time.time() - t0)}
    rows.append(r); print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
CV = pd.DataFrame(rows).set_index("fold"); CV.loc["mean"] = CV.mean()
print(CV.round(4).to_string())

In [ ]:
#@title Ablation внешних источников и выбор групп по данным (критерий 2а)
res = {"A (без внешних)": CV.loc["mean", "A (база)"], "A×B (все)": CV.loc["mean", "A×B (все источники)"]}
per_group = {}
for g in tqdm(list(GROUPS), desc="ablation"):
    sc = [wape_score(p.y, p.pred) for p in (fold_eval([x for x in GROUPS if x != g], *f[1:])[1] for f in FOLDS)]
    res[f"без {g}"] = float(np.mean(sc)); per_group[g] = res["A×B (все)"] - res[f"без {g}"]
    print(f"без {g}: {res[f'без {g}']:.4f}   вклад источника {per_group[g]:+.4f}")
ABL = pd.Series(res).round(4).to_frame("mean WAPE-score"); print(ABL.to_string())
SELECTED = [g for g in GROUPS if per_group[g] > 0]
print("\nИсточники в финальной модели (вклад на валидации > 0):", SELECTED)
if SELECTED != list(GROUPS):
    rows = []
    for name, cut, a, b in FOLDS:
        _, p = fold_eval(SELECTED, cut, a, b); FOLD_PRED[name] = p; rows.append(wape_score(p.y, p.pred))
    CV["A×B (выбранные)"] = rows + [float(np.mean(rows))]
else:
    CV["A×B (выбранные)"] = CV["A×B (все источники)"]
print(CV.round(4).to_string())

In [ ]:
#@title Коэффициенты K, вычисленные моделью (что увидит диспетчер в UI)
p = FOLD_PRED["F4 июн→июн–авг"]
kc = [f"K_{g}" for g in GROUPS]
day = p.groupby("date").apply(lambda g: pd.Series({c: np.average(g[c], weights=np.maximum(g.base, 1e-6)) for c in kc}))
day.plot(figsize=(16, 4), title="K по источникам по дням (F4: 12–13 июня — праздники)"); plt.axhline(1, c="gray", ls="--"); plt.show()
print(day.loc["2025-06-10":"2025-06-16"].round(3))

In [ ]:
#@title Ошибка по маршрутам и по горизонту
p = FOLD_PRED["F2 сен→окт"]
print(p.groupby("route").apply(lambda g: wape_score(g.y, g.pred) if g.y.sum() else np.nan).round(3).to_frame("WAPE-score").T)
p1 = FOLD_PRED["F1 авг→сен–окт"]
d = p1.groupby("date").apply(lambda g: wape_score(g.y, g.pred))
ax = d.plot(figsize=(16, 3.5), title="F1: дневной WAPE-score по горизонту 1…61 день (прямой прогноз)"); ax.axhline(0.88, ls="--", c="gray"); plt.show()

## 🔄 Потоковый режим

`StreamingForecaster.ingest(день)` добавляет суточный агрегат валидаций `route × hour`; признаки A пересчитываются от новой точки прогноза. `forecast(h)` отдаёт прогноз на h дней с разложением `base × K`. Переобучение — по расписанию (`refit_every` дней).

Прогон: модель обучена по 31.08, сентябрь–октябрь подаются день за днём, после каждого дня — прогноз на 1 и 7 дней вперёд.

In [ ]:
#@title StreamingForecaster + прогон день за днём
class StreamingForecaster:
    def __init__(self, model, Y_init, refit_every=None):
        self.model, self.Y = model, Y_init.copy()
        self.refit_every, self.n = refit_every, 0

    @property
    def last_idx(self): return self.Y.shape[1] - 1

    def ingest(self, day_df):
        day = np.zeros((len(ROUTES), 1, 24), dtype=np.float32)
        if len(day_df):
            day[day_df.route.map(R_IDX).values, 0, day_df.hour.values] = day_df.boardings.values
        self.Y = np.concatenate([self.Y, day], axis=1); self.n += 1
        if self.refit_every and self.n % self.refit_every == 0:
            self.model = TramForecaster(self.model.groups).fit(self.Y, self.last_idx)

    def forecast(self, horizon_days, weather=None):
        o = self.last_idx
        return self.model.predict(self.Y, o, range(o + 1, o + 1 + horizon_days), weather=weather)

cut = D_IDX[pd.Timestamp("2025-08-31")]; end = D_IDX[pd.Timestamp(HIST_END)]
sf = StreamingForecaster(TramForecaster(SELECTED).fit(Y[:, :cut + 1], cut), Y[:, :cut + 1])
lab_by_day = {d: g for d, g in lab.groupby("date")}
res, lat = [], []
for t in tqdm(range(cut + 1, end + 1), desc="stream"):
    t0 = time.perf_counter(); fc = sf.forecast(7, weather=W_TRAIN); lat.append((time.perf_counter() - t0) * 1000)
    fc = fc[fc.date.map(D_IDX).values <= end]
    fc["y"] = Y[fc.route.map(R_IDX).values, fc.date.map(D_IDX).values, fc.hour.values]
    d1 = fc[fc.date == ALL_DATES[t]]
    res.append({"date": ALL_DATES[t], "day-ahead": wape_score(d1.y, d1.pred), "1–7 дней": wape_score(fc.y, fc.pred)})
    sf.ingest(lab_by_day.get(ALL_DATES[t], pd.DataFrame(columns=["route", "hour", "boardings"])))
    if len(res) % 10 == 0:
        print(f"{ALL_DATES[t].date()}  day-ahead WAPE-score за 10 дней = {np.mean([r['day-ahead'] for r in res[-10:]]):.4f}")
S = pd.DataFrame(res).set_index("date")
print(f"\nПотоковый режим сен–окт: day-ahead = {S['day-ahead'].mean():.4f}, 1–7 дней = {S['1–7 дней'].mean():.4f}")
print(f"latency forecast(7 дней × 10 маршрутов × 24 ч): p50 = {np.percentile(lat, 50):.0f} мс, p95 = {np.percentile(lat, 95):.0f} мс")
S.plot(figsize=(16, 3.5), title="Потоковый режим: WAPE-score"); plt.show()

## 🚀 Финальная модель на всей истории, прогноз ноя–дек, артефакты, сабмит

In [ ]:
#@title Обучение на янв–окт и прогноз 01.11–31.12
ci = D_IDX[pd.Timestamp(HIST_END)]
FINAL = TramForecaster(SELECTED).fit(Y, ci)
FC = FINAL.predict(Y, ci, range(D_IDX[pd.Timestamp(FORECAST_START)], D_IDX[pd.Timestamp(FORECAST_END)] + 1), weather=W_FC)
kc = [f"K_{g}" for g in GROUPS]
print("Прогноз, тыс. посадок:"); print((FC.groupby([FC.date.dt.month, "route"]).pred.sum().unstack() / 1e3).round(0))
wk = lambda g: pd.Series({c: np.average(g[c], weights=np.maximum(g.base, 1e-6)) for c in kc})
print("\nK в ключевые дни (вычислены моделью B):")
print(FC.groupby("date").apply(wk).loc[["2025-11-01", "2025-11-03", "2025-11-04", "2025-11-05", "2025-12-30", "2025-12-31"]].round(3))
hist = pd.Series(Y.sum((0, 2)), index=HIST_DATES)["2025-09-01":]
ax = hist.plot(figsize=(16, 4), label="факт"); FC.groupby("date").pred.sum().plot(ax=ax, label="прогноз"); ax.legend(); plt.show()

In [ ]:
#@title 💾 Артефакты для бэкенда (контракт) + ONNX + сабмит
stamp = time.strftime("%Y%m%d_%H%M")
FINAL.A.save_model(str(OUT / "model_A_base.txt"))
if FINAL.B is not None: FINAL.B.save_model(str(OUT / "model_B_external.txt"))
contract = {
    "model_version": f"lgbm_AxB_{stamp}",
    "A": {"features": A_FEATS, "target": f"y/({A_REF}+1)", "output": f"pred_ratio*({A_REF}+1)", "train_max_h": A_TRAIN_MAX_H},
    "B": {"features": getattr(FINAL, "cols", []), "groups_selected": SELECTED, "neutral_values": {k: (None if isinstance(v, float) and np.isnan(v) else v) for k, v in NEUTRAL.items()}},
    "active_hours": {int(r): [int(h) for h in range(24) if ACTIVE[i, h]] for i, r in enumerate(ROUTES)},
    "output": "pred = base * K_calendar * K_weather * K_events * K_traffic; K=1 при недоступном источнике; округление half-up только при выгрузке",
    "cv": CV.round(4).to_dict(), "ablation": ABL["mean WAPE-score"].to_dict(),
    "streaming": {"day_ahead": float(S["day-ahead"].mean()), "week": float(S["1–7 дней"].mean()), "latency_p95_ms": float(np.percentile(lat, 95))},
    "weather_for_forecast": WEATHER_FOR_FORECAST, "seed": SEED, "lightgbm": lgb.__version__,
}
json.dump(contract, open(OUT / "contract.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
FC.assign(date=FC.date.dt.strftime("%Y-%m-%d")).to_csv(OUT / "forecast_components.csv", sep=";", index=False)
try:
    import onnxmltools
    from onnxmltools.convert.common.data_types import FloatTensorType
    for name, m, feats in [("A_base", FINAL.A, A_FEATS), ("B_external", FINAL.B, getattr(FINAL, "cols", []))]:
        if m is None: continue
        onx = onnxmltools.convert_lightgbm(m, initial_types=[("input", FloatTensorType([None, len(feats)]))], target_opset=15)
        open(OUT / f"model_{name}.onnx", "wb").write(onx.SerializeToString())
    print("ONNX: сохранено")
except Exception as e:
    print("ONNX пропущен:", type(e).__name__, e)

sub = template[["route", "date", "hour"]].copy(); sub["date"] = pd.to_datetime(sub.date)
sub = sub.merge(FC[["route", "date", "hour", "pred"]], on=["route", "date", "hour"], how="left")
sub["prediction"] = np.floor(sub.pred.fillna(0).clip(lower=0) + 0.5).astype(int)   # half-up только на выгрузке
sub["date"] = sub.date.dt.strftime("%Y-%m-%d"); sub = sub[["route", "date", "hour", "prediction"]]
assert len(sub) == 14640 and not sub.duplicated(["route", "date", "hour"]).any() and (sub.prediction >= 0).all()
cvm = CV.loc["mean", "A×B (выбранные)"]
fname = SUB_DIR / f"submission_ml_{stamp}_cv{cvm:.4f}.csv"
sub.to_csv(fname, sep=";", index=False, encoding="utf-8"); sub.to_csv(SUB_DIR / "submission_ml_latest.csv", sep=";", index=False)
print(f"✅ {fname}\n   артефакты: {sorted(p.name for p in OUT.iterdir())}\n   CV = {cvm:.4f} | streaming day-ahead = {S['day-ahead'].mean():.4f}")
if IN_COLAB:
    try:
        from google.colab import files; files.download(str(fname))
    except Exception as e: print(e)